## Implement Semantic Search on PostgreSQL

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv openai==2.38.0

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
from psycopg.rows import dict_row


load_dotenv()

# Loading the database configurations
host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")
password = os.getenv("DATABASE_PASSWORD")

# Loading the Azure OpenAI configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")

### Create the Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    azure_endpoint=azure_openai_endpoint,
    api_version="2024-06-01"
)

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    response = azure_openai_client.embeddings.create(
        model=embedding_model_name,
        input=text
    )

    return response.data[0].embedding

### Generate Vector Embeddings for the User Query

In [ ]:
user_query = "How is GreenSteel reducing emissions?"

query_embedding = generate_embeddings(user_query)

### Implement a Vector Search Query

In [ ]:
search_query = """
SELECT
    ChunkID,
    CompanyName,
    ChunkText,
    ChunkEmbedding <=> %s::vector AS distance
FROM RAG.ESG_Chunks
ORDER BY ChunkEmbedding <=> %s::vector
LIMIT 5
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,
                query_embedding
            )
        )

        results = cur.fetchall()

for result in results:

    print("company name: {}".format(result["companyname"]))
    print("vector distance: {}".format(result["distance"]))
    print("chunk text: {}".format(result["chunktext"]))
    print("================================")

### Metadata Filtering + Vector Search

In [ ]:
search_query = """
SELECT
    ChunkID,
    CompanyName,
    ChunkText,
    ChunkEmbedding <=> %s::vector AS distance
FROM RAG.ESG_Chunks
WHERE CompanyName = 'GreenSteel Ltd'
ORDER BY ChunkEmbedding <=> %s::vector
LIMIT 5
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,
                query_embedding
            )
        )

        results = cur.fetchall()

for result in results:

    print("company name: {}".format(result["companyname"]))
    print("vector distance: {}".format(result["distance"]))
    print("chunk text: {}".format(result["chunktext"]))
    print("================================")

### Create a GIN Index for Keyword Search

In [ ]:
create_fts_index_query = """
CREATE INDEX IF NOT EXISTS idx_esg_chunks_fts
ON RAG.ESG_Chunks
USING GIN
(
    to_tsvector('english', ChunkText)
);
"""

with pool.connection() as conn:

    with conn.cursor() as cur:

        cur.execute(create_fts_index_query)

    conn.commit()

print("Full Text Search Index Created Successfully")

### Implement Hybrid Search

In [ ]:
search_query = """
SELECT

    ChunkID,
    CompanyName,

    (
        (
            1 -
            (
                ChunkEmbedding <=> %s::vector
            )
        ) * 0.7

        +

        ts_rank(
            to_tsvector(
                'english',
                ChunkText
            ),
            plainto_tsquery(
                'english',
                %s
            )
        ) * 0.3

    ) AS hybrid_score,

    ChunkText

FROM RAG.ESG_Chunks

WHERE

    to_tsvector(
        'english',
        ChunkText
    )

    @@

    plainto_tsquery(
        'english',
        %s
    )

    OR

    (
        ChunkEmbedding <=> %s::vector
    ) < 0.5

ORDER BY hybrid_score DESC

LIMIT 10
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,  # ChunkEmbedding <=> %s::vector
                user_query,    # plainto_tsquery text
                user_query,    # plainto_tsquery text
                query_embedding   # ChunkEmbedding <=> %s::vector
            )
        )

        results = cur.fetchall()

for result in results:

    print(f"Company: {result['companyname']}")
    print(f"Hybrid Score: {result['hybrid_score']:.4f}")
    print("--------------------------------------")
    print(result["chunktext"])
    print("======================================\n")